# 03. NV3: Mô phỏng và Đánh giá chính sách phí nhàn rỗi (Idle Fee Policy)

## Mục tiêu nghiên cứu (NV3)
Đánh giá tác động kinh tế và hành vi của các kịch bản áp dụng phí nhàn rỗi (Overstay/Idle Fee):
1. **Kịch bản cơ sở (Baseline):** Không thu phí nhàn rỗi.
2. **Kịch bản 1 (Phí bậc thang đơn giản):** Miễn phí 30 phút đầu sau khi sạc xong (Grace period = 30m), sau đó tính phí \$0.20/phút.
3. **Kịch bản 2 (Phí lũy tiến nghiêm ngặt):** Miễn phí 15 phút đầu, 15–60 phút tính \$0.40/phút, sau 60 phút tính \$1.00/phút.
4. Ước tính doanh thu bổ sung cho đơn vị vận hành trạm và mức độ giảm thiểu thời gian chiếm giữ lãng phí.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.config import SESSIONS_FEATURES_PARQUET
from src.visualization import set_plotting_style

set_plotting_style()
df = pd.read_parquet(SESSIONS_FEATURES_PARQUET)
valid = df[df['is_valid_session']].copy()
print(f"Số lượng phiên sạc hợp lệ đưa vào mô phỏng: {len(valid):,}")


### 1. Hàm mô phỏng phí nhàn rỗi theo các kịch bản chính sách


In [ ]:
def simulate_idle_fees(df_in, grace_period_min=30, rate_per_min=0.20):
    idle_min = df_in['idle_time_sec'] / 60.0
    billable_min = np.maximum(0, idle_min - grace_period_min)
    fee = billable_min * rate_per_min
    return billable_min, fee

# Kịch bản 1: Grace 30 phút, $0.20/phút
valid['billable_min_scen1'], valid['idle_fee_scen1'] = simulate_idle_fees(valid, grace_period_min=30, rate_per_min=0.20)

# Kịch bản 2: Grace 15 phút, $0.50/phút
valid['billable_min_scen2'], valid['idle_fee_scen2'] = simulate_idle_fees(valid, grace_period_min=15, rate_per_min=0.50)

print("Tổng kết mô phỏng doanh thu phí nhàn rỗi:")
print(f" - Kịch bản 1 (30m grace, $0.20/m): Tổng thu ${valid['idle_fee_scen1'].sum():,.2f}")
print(f" - Kịch bản 2 (15m grace, $0.50/m): Tổng thu ${valid['idle_fee_scen2'].sum():,.2f}")


### 2. So sánh tác động tài chính và độ nhạy cảm của người dùng theo kịch bản


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scen_comparison = pd.DataFrame({
    'Chính sách': ['Hiện tại (Không phí)', 'Kịch bản 1 (30m grace, $0.20/m)', 'Kịch bản 2 (15m grace, $0.50/m)'],
    'Tổng doanh thu phí ($)': [0, valid['idle_fee_scen1'].sum(), valid['idle_fee_scen2'].sum()]
})
sns.barplot(data=scen_comparison, x='Chính sách', y='Tổng doanh thu phí ($)', ax=ax, palette='Blues_d')
ax.set_title('Ước tính Tổng doanh thu phí nhàn rỗi theo từng kịch bản chính sách')
plt.show()
